In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [47]:
from fredapi import Fred

fred = Fred(
    api_key="c218530425a9461b7eed4068455bb771 "
)

In [48]:
# Unemployment Rate
unrate = fred.get_series(
    "UNRATE"
)

In [49]:
# Federal Funds Rate
fedfunds = fred.get_series(
    "FEDFUNDS"
)

In [50]:
# Inflation Rate
cpi = fred.get_series(
    "CPIAUCSL"
)

In [51]:
# Building Macro Dataset
macro_df = pd.DataFrame({
    "unrate": unrate,
    "fedfunds": fedfunds,
    "cpi": cpi
})

macro_df = macro_df.reset_index()

In [52]:
#Rename
macro_df.columns = [
    "date",
    "unrate",
    "fedfunds",
    "cpi"
]

In [53]:
# Load risk_df (from 01_EDA pipeline output)
from pathlib import Path

candidates = [
    Path('..') / 'Data' / 'processed' / 'risk_model_base.csv',
    Path('..') / 'data' / 'processed' / 'risk_model_base.csv',
]

risk_df = pd.read_csv(str(next(p for p in candidates if p.exists())))
print(f"Loaded risk_df with shape: {risk_df.shape}")

Loaded risk_df with shape: (59494, 152)


In [54]:
"issue_d" in risk_df.columns

True

In [55]:
# Convert issue_d to datetime
risk_df["issue_d"] = pd.to_datetime(
    risk_df["issue_d"]
)

C:\Users\BOSS\AppData\Local\Temp\ipykernel_9532\1650868535.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  risk_df["issue_d"] = pd.to_datetime(


In [56]:
risk_df["issue_month"] = (
    risk_df["issue_d"]
    .dt.to_period("M")
)

C:\Users\BOSS\AppData\Local\Temp\ipykernel_9532\2074779530.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  risk_df["issue_month"] = (


In [57]:
macro_df["issue_month"] = (
    pd.to_datetime(
        macro_df["date"]
    )
    .dt.to_period("M")
)

In [58]:
risk_macro_df = risk_df.merge(
    macro_df,
    on="issue_month",
    how="left"
)

In [59]:
risk_macro_df.to_csv(
    "../data/processed/risk_macro_df.csv",
    index=False
)

In [60]:
risk_macro_df[
    [
        "issue_d",
        "unrate",
        "fedfunds",
        "cpi"
    ]
].head()

,issue_d,unrate,fedfunds,cpi
0,2014-05-01,6.3,0.09,236.918
1,2015-04-01,5.4,0.12,236.222
2,2017-12-01,4.1,1.30,247.805
3,2016-08-01,4.9,0.40,240.545
4,2015-04-01,5.4,0.12,236.222


In [61]:
risk_macro_df[
    ["unrate", "fedfunds", "cpi"]
].isnull().sum()

unrate      0
fedfunds    0
cpi         0
dtype: int64

In [62]:
macro_features = [
    "unrate",
    "fedfunds",
    "cpi"
]

In [63]:
model_macro_df = risk_macro_df[
    [
        "loan_amnt",
        "term",
        "int_rate",
        "grade",
        "sub_grade",
        "emp_length",
        "home_ownership",
        "annual_inc",
        "verification_status",
        "purpose",
        "dti",
        "fico_range_low",
        "fico_range_high",
        "open_acc",
        "revol_bal",
        "revol_util",
        "delinq_2yrs",
        "pub_rec",
        "inq_last_6mths",
        "unrate",
        "fedfunds",
        "cpi",
        "target"
    ]
].copy()

In [64]:
model_macro_df.shape

(59494, 23)

In [65]:
model_macro_df.isnull().sum()

loan_amnt                 0
term                      0
int_rate                  0
grade                     0
sub_grade                 0
emp_length             3470
home_ownership            0
annual_inc                0
verification_status       0
purpose                   0
dti                      20
fico_range_low            0
fico_range_high           0
open_acc                  0
revol_bal                 0
revol_util               32
delinq_2yrs               0
pub_rec                   0
inq_last_6mths            0
unrate                    0
fedfunds                  0
cpi                       0
target                    0
dtype: int64

In [66]:
import scorecardpy as sc

iv_macro = sc.iv(
    dt=model_macro_df,
    y="target"
)

iv_macro.sort_values(
    by="info_value",
    ascending=False
)

,variable,info_value
1,int_rate,0.569668
7,sub_grade,0.496593
10,grade,0.455679
18,dti,0.439362
14,revol_bal,0.380001
13,annual_inc,0.316466
8,loan_amnt,0.196713
11,term,0.183950
5,revol_util,0.151803
6,fico_range_high,0.131421


In [67]:
# fill missing for object columns
cat_cols = model_macro_df.select_dtypes(include=["object"]).columns.tolist()
if cat_cols:
    model_macro_df[cat_cols] = model_macro_df[cat_cols].fillna("missing")

# exclude obvious identifier/date/url/verbose text columns and very high-cardinality cols
exclude_patterns = [
    'id', 'zip', 'url', 'desc', 'title', 'date', 'month', 'issue_d',
    'last_pymnt_d', 'last_credit_pull_d', 'earliest_cr_line'
]
high_card_threshold = 100
high_cardinality = [
    c for c in model_macro_df.columns
    if model_macro_df[c].nunique(dropna=True) > high_card_threshold
]
exclude_cols = [c for c in model_macro_df.columns if any(p in c.lower() for p in exclude_patterns)]
# ensure target not excluded
excluded = [c for c in set(high_cardinality + exclude_cols) if c != 'target']

x_cols = [c for c in model_macro_df.columns if c != 'target' and c not in excluded]

print(f"Excluded {len(excluded)} columns (examples): {excluded[:10]}")
print(f"Using {len(x_cols)} variables for woebin")

# run woebin non-interactively (skip the categorical-unique prompt)
bins_macro = sc.woebin(
    model_macro_df,
    y="target",
    x=x_cols,
    check_cate_num=False
)

print('woebin completed for', len(bins_macro), 'variables')

C:\Users\BOSS\AppData\Local\Temp\ipykernel_9532\4101321982.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = model_macro_df.select_dtypes(include=["object"]).columns.tolist()


Excluded 7 columns (examples): ['annual_inc', 'int_rate', 'revol_bal', 'revol_util', 'cpi', 'dti', 'loan_amnt']
Using 15 variables for woebin
[INFO] creating woe binning ...
woebin completed for 15 variables


In [68]:
import traceback

# Ensure bins_macro exists
if 'bins_macro' not in globals():
    raise NameError("bins_macro not found — run the woebin cell first")

# Fill object columns if any (defensive)
cat_cols = model_macro_df.select_dtypes(include=['object']).columns.tolist()
if cat_cols:
    model_macro_df[cat_cols] = model_macro_df[cat_cols].fillna('missing')

# Filter bins to variables that exist in model_macro_df
all_bin_vars = list(bins_macro.keys())
present_bin_vars = [c for c in all_bin_vars if c in model_macro_df.columns]
removed_bins = [c for c in all_bin_vars if c not in model_macro_df.columns]
if removed_bins:
    print(f"Removing {len(removed_bins)} bins not present in dataframe: {removed_bins[:10]}")

if not present_bin_vars:
    raise ValueError('No bin variables match columns in model_macro_df')

filtered_bins = {k: bins_macro[k] for k in present_bin_vars}

# Apply WOE transformation
try:
    model_macro_woe = sc.woebin_ply(model_macro_df, filtered_bins)
    print('model_macro_woe shape:', getattr(model_macro_woe, 'shape', None))
except Exception:
    traceback.print_exc()
    model_macro_woe = None
    print('model_macro_woe not created due to error')

[INFO] converting into woe values ...


C:\Users\BOSS\AppData\Local\Temp\ipykernel_9532\4247168969.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = model_macro_df.select_dtypes(include=['object']).columns.tolist()


model_macro_woe not created due to error


Traceback (most recent call last):
  File "C:\Users\BOSS\AppData\Local\Temp\ipykernel_9532\4247168969.py", line 26, in <module>
    model_macro_woe = sc.woebin_ply(model_macro_df, filtered_bins)
  File "c:\Users\BOSS\OneDrive\Desktop\Credit Card Score Builder\venv\Lib\site-packages\scorecardpy\woebin.py", line 1134, in woebin_ply
    if replace_blank: dt = rep_blank_na(dt)
                           ~~~~~~~~~~~~^^^^
  File "c:\Users\BOSS\OneDrive\Desktop\Credit Card Score Builder\venv\Lib\site-packages\scorecardpy\condition_fun.py", line 87, in rep_blank_na
    dat[col].astype(str).str.findall(r'^\s*$').apply(lambda x: 0 if len(x) == 0 else 1).sum() > 0]
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\BOSS\OneDrive\Desktop\Credit Card Score Builder\venv\Lib\site-packages\pandas\core\series.py", line 5084, in apply
    ).apply()
      ~~~~~^^
  File "c:\Users\BOSS\OneDrive\Desktop\Credit Card Score Builder\venv\Lib\site-packages\p

In [72]:
model_macro_df.dtypes

loan_amnt              float64
term                       str
int_rate               float64
grade                      str
sub_grade                  str
emp_length                 str
home_ownership             str
annual_inc             float64
verification_status        str
purpose                    str
dti                    float64
fico_range_low         float64
fico_range_high        float64
open_acc               float64
revol_bal              float64
revol_util             float64
delinq_2yrs            float64
pub_rec                float64
inq_last_6mths         float64
unrate                 float64
fedfunds               float64
cpi                    float64
target                   int64
dtype: object

In [73]:
cat_cols = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose"
]

for col in cat_cols:
    print("\n", col)
    print(model_macro_df[col].apply(type).value_counts())


 term
term
<class 'str'>    59494
Name: count, dtype: int64

 grade
grade
<class 'str'>    59494
Name: count, dtype: int64

 sub_grade
sub_grade
<class 'str'>    59494
Name: count, dtype: int64

 emp_length
emp_length
<class 'str'>    59494
Name: count, dtype: int64

 home_ownership
home_ownership
<class 'str'>    59494
Name: count, dtype: int64

 verification_status
verification_status
<class 'str'>    59494
Name: count, dtype: int64

 purpose
purpose
<class 'str'>    59494
Name: count, dtype: int64


In [74]:
print(bins_macro.keys())

dict_keys(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'fico_range_low', 'fico_range_high', 'open_acc', 'delinq_2yrs', 'pub_rec', 'inq_last_6mths', 'unrate', 'fedfunds'])


In [75]:
for var in bins_macro.keys():
    try:
        temp = sc.woebin_ply(
            model_macro_df[[var, "target"]],
            {var: bins_macro[var]}
        )
        print(f"SUCCESS: {var}")
    except Exception as e:
        print(f"FAILED: {var}")
        print(e)
        break

[INFO] converting into woe values ...
SUCCESS: term
[INFO] converting into woe values ...
SUCCESS: grade
[INFO] converting into woe values ...
SUCCESS: sub_grade
[INFO] converting into woe values ...
SUCCESS: emp_length
[INFO] converting into woe values ...
SUCCESS: home_ownership
[INFO] converting into woe values ...
SUCCESS: verification_status
[INFO] converting into woe values ...
SUCCESS: purpose
[INFO] converting into woe values ...
SUCCESS: fico_range_low
[INFO] converting into woe values ...
SUCCESS: fico_range_high
[INFO] converting into woe values ...
SUCCESS: open_acc
[INFO] converting into woe values ...
SUCCESS: delinq_2yrs
[INFO] converting into woe values ...
SUCCESS: pub_rec
[INFO] converting into woe values ...
SUCCESS: inq_last_6mths
[INFO] converting into woe values ...
SUCCESS: unrate
[INFO] converting into woe values ...
SUCCESS: fedfunds


In [76]:
import numpy as np

numeric_cols = model_macro_df.select_dtypes(
    include=np.number
).columns

for col in numeric_cols:
    inf_count = np.isinf(
        model_macro_df[col]
    ).sum()

    if inf_count > 0:
        print(col, inf_count)

In [77]:
model_macro_df.isnull().sum().sort_values(
    ascending=False
).head(20)

revol_util             32
dti                    20
loan_amnt               0
grade                   0
sub_grade               0
term                    0
int_rate                0
home_ownership          0
emp_length              0
verification_status     0
annual_inc              0
fico_range_low          0
fico_range_high         0
open_acc                0
purpose                 0
revol_bal               0
delinq_2yrs             0
pub_rec                 0
inq_last_6mths          0
unrate                  0
dtype: int64

In [78]:
model_macro_df["dti"] = (
    model_macro_df["dti"]
    .fillna(model_macro_df["dti"].median())
)

model_macro_df["revol_util"] = (
    model_macro_df["revol_util"]
    .fillna(model_macro_df["revol_util"].median())
)

In [79]:
model_macro_df.isnull().sum().sort_values(
    ascending=False
).head(10)

loan_amnt              0
term                   0
int_rate               0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
purpose                0
dtype: int64

In [87]:
# ============================================================
# MACRO ENHANCED CREDIT RISK SCORECARD
# ============================================================

import os
import pandas as pd
import numpy as np
import scorecardpy as sc
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

# determine project root (one level up from the notebooks folder)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
OUT_DIR = os.path.join(PROJECT_ROOT, "Data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# ============================================================
# LOAD MACRO ENHANCED DATASET
# ============================================================

model_macro_df = risk_macro_df[
    [
        "loan_amnt",
        "term",
        "int_rate",
        "grade",
        "sub_grade",
        "emp_length",
        "home_ownership",
        "annual_inc",
        "verification_status",
        "purpose",
        "dti",
        "fico_range_low",
        "fico_range_high",
        "open_acc",
        "revol_bal",
        "revol_util",
        "delinq_2yrs",
        "pub_rec",
        "inq_last_6mths",
        "unrate",
        "fedfunds",
        "cpi",
        "target"
    ]
].copy()

print(model_macro_df.shape)


# ============================================================
# MISSING VALUE TREATMENT
# ============================================================

model_macro_df["emp_length"] = (
    model_macro_df["emp_length"]
    .fillna("Missing")
)

model_macro_df["dti"] = (
    model_macro_df["dti"]
    .fillna(
        model_macro_df["dti"].median()
    )
)

model_macro_df["revol_util"] = (
    model_macro_df["revol_util"]
    .fillna(
        model_macro_df["revol_util"].median()
    )
)

model_macro_df["unrate"] = (
    model_macro_df["unrate"]
    .fillna(
        model_macro_df["unrate"].median()
    )
)

model_macro_df["fedfunds"] = (
    model_macro_df["fedfunds"]
    .fillna(
        model_macro_df["fedfunds"].median()
    )
)

model_macro_df["cpi"] = (
    model_macro_df["cpi"]
    .fillna(
        model_macro_df["cpi"].median()
    )
)

print(model_macro_df.isnull().sum())


# ============================================================
# INFORMATION VALUE
# ============================================================

iv_macro = sc.iv(
    dt=model_macro_df,
    y="target"
)

iv_macro = iv_macro.sort_values(
    by="info_value",
    ascending=False
)

print(iv_macro)

iv_macro.to_csv(
    os.path.join(OUT_DIR, "macro_information_value.csv"),
    index=False
)


# ============================================================
# WOE BINNING
# ============================================================

bins_macro = sc.woebin(
    model_macro_df,
    y="target"
)

joblib.dump(
    bins_macro,
    os.path.join(MODELS_DIR, "macro_woe_bins.pkl")
)


# ============================================================
# WOE TRANSFORMATION
# ============================================================

model_macro_woe = sc.woebin_ply(
    model_macro_df,
    bins_macro
)

print(model_macro_woe.shape)

model_macro_woe.to_csv(
    os.path.join(OUT_DIR, "model_macro_woe.csv"),
    index=False
)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    model_macro_woe,
    test_size=0.30,
    random_state=42,
    stratify=model_macro_woe["target"]
)

print(train_df.shape)
print(test_df.shape)


# ============================================================
# FEATURES & TARGET
# ============================================================

X_train = train_df.drop(
    columns=["target"]
)

y_train = train_df["target"]

X_test = test_df.drop(
    columns=["target"]
)

y_test = test_df["target"]


# ============================================================
# LOGISTIC REGRESSION
# ============================================================

lr_macro = LogisticRegression(
    max_iter=1000
)

lr_macro.fit(
    X_train,
    y_train
)

joblib.dump(
    lr_macro,
    os.path.join(MODELS_DIR, "macro_logistic_model.pkl")
)


# ============================================================
# PREDICTIONS
# ============================================================

pred_prob_macro = (
    lr_macro.predict_proba(
        X_test
    )[:,1]
)


# ============================================================
# ROC AUC
# ============================================================

auc_macro = roc_auc_score(
    y_test,
    pred_prob_macro
)

print(
    f"Macro Model AUC: {auc_macro:.4f}"
)


# ============================================================
# KS STATISTIC
# ============================================================

good = pred_prob_macro[
    y_test == 0
]

bad = pred_prob_macro[
    y_test == 1
]

ks_macro = ks_2samp(
    good,
    bad
)

print(
    f"Macro Model KS: {ks_macro.statistic:.4f}"
)


# ============================================================
# MODEL COEFFICIENTS
# ============================================================

coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lr_macro.coef_[0]
})

coef_df = coef_df.sort_values(
    by="coefficient",
    ascending=False
)

print(coef_df)

coef_df.to_csv(
    os.path.join(OUT_DIR, "macro_model_coefficients.csv"),
    index=False
)


# ============================================================
# SCORING DATASET
# ============================================================

test_results = pd.DataFrame({
    "actual": y_test,
    "probability_of_default": pred_prob_macro
})


# ============================================================
# RISK BANDS
# ============================================================

test_results["risk_band"] = pd.qcut(
    test_results["probability_of_default"],
    q=5,
    labels=[
        "Very Low Risk",
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Very High Risk"
    ]
)


# ============================================================
# DECISION ENGINE
# ============================================================

def decision(pd_value):

    if pd_value < 0.10:
        return "Approve"

    elif pd_value < 0.25:
        return "Manual Review"

    return "Reject"


test_results["decision"] = (
    test_results["probability_of_default"]
    .apply(decision)
)


# ============================================================
# CREDIT SCORE
# ============================================================

test_results["credit_score"] = (
    850
    -
    (
        test_results[
            "probability_of_default"
        ] * 550
    )
)

test_results["credit_score"] = (
    test_results["credit_score"]
    .round()
    .astype(int)
)


# ============================================================
# SAVE RESULTS
# ============================================================

test_results.to_csv(
    os.path.join(OUT_DIR, "macro_scored_loans.csv"),
    index=False
)

print(
    test_results.head()
)


# ============================================================
# BASELINE VS MACRO COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Macro Enhanced"
    ],
    "AUC": [
        0.70,
        auc_macro
    ],
    "KS": [
        0.30,
        ks_macro.statistic
    ]
})

print(comparison)

comparison.to_csv(
    os.path.join(OUT_DIR, "model_comparison.csv"),
    index=False
)

print("Macro Enhanced Pipeline Complete")


(59494, 23)
loan_amnt              0
term                   0
int_rate               0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
purpose                0
dti                    0
fico_range_low         0
fico_range_high        0
open_acc               0
revol_bal              0
revol_util             0
delinq_2yrs            0
pub_rec                0
inq_last_6mths         0
unrate                 0
fedfunds               0
cpi                    0
target                 0
dtype: int64
               variable  info_value
1              int_rate    0.569668
7             sub_grade    0.496593
10                grade    0.455679
18                  dti    0.439192
14            revol_bal    0.380001
13           annual_inc    0.316466
8             loan_amnt    0.196713
11                 term    0.183950
5            revol_util    0.151781
6       fico_range_high    0.131421
21  

In [ ]:
print(lr_macro.feature_names_in_)